In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("ews").getOrCreate()

info = spark.read.csv("../data/raw/studentInfo.csv", header=True, inferSchema=True)
print(info.count())
info.groupBy("final_result").count().show()

32593
+------------+-----+
|final_result|count|
+------------+-----+
|        Fail| 7052|
| Distinction| 3024|
|   Withdrawn|10156|
|        Pass|12361|
+------------+-----+



In [2]:
from pyspark.sql import functions as F

info = info.withColumn("at_risk",
    F.when(F.col("final_result").isin("Fail", "Withdrawn"), 1).otherwise(0))

info.groupBy("at_risk").count().show()
info.groupBy("code_module", "at_risk").count().orderBy("code_module").show(20)

+-------+-----+
|at_risk|count|
+-------+-----+
|      1|17208|
|      0|15385|
+-------+-----+

+-----------+-------+-----+
|code_module|at_risk|count|
+-----------+-------+-----+
|        AAA|      1|  217|
|        AAA|      0|  531|
|        BBB|      1| 4155|
|        BBB|      0| 3754|
|        CCC|      1| 2756|
|        CCC|      0| 1678|
|        DDD|      1| 3662|
|        DDD|      0| 2610|
|        EEE|      1| 1284|
|        EEE|      0| 1650|
|        FFF|      0| 3648|
|        FFF|      1| 4114|
|        GGG|      1| 1020|
|        GGG|      0| 1514|
+-----------+-------+-----+



In [3]:
reg = spark.read.csv("../data/raw/studentRegistration.csv", header=True, inferSchema=True)

for day in [28, 56, 84]:
    n = reg.filter(F.col("date_unregistration") <= day).count()
    print(f"unregistered by day {day}: {n}")

unregistered by day 28: 5055
unregistered by day 56: 6071
unregistered by day 84: 6869


In [4]:
keys = ["code_module", "code_presentation", "id_student"]
base = info.join(reg.select(*keys, "date_unregistration"), keys, "left")

def active_cohort(day):
    return base.filter(F.col("date_unregistration").isNull() |
                       (F.col("date_unregistration") > day))

for day in [28, 56, 84]:
    c = active_cohort(day)
    rate = c.agg(F.avg("at_risk")).first()[0]
    print(f"day {day}: {c.count()} students, at-risk rate {rate:.1%}")

day 28: 27538 students, at-risk rate 44.1%
day 56: 26522 students, at-risk rate 42.0%
day 84: 25724 students, at-risk rate 40.2%


In [5]:
svle = spark.read.csv("../data/raw/studentVle.csv", header=True, inferSchema=True)
print(svle.count())

early = (svle.filter(F.col("date") <= 28)
         .groupBy(keys).agg(F.sum("sum_click").alias("clicks_28")))

cohort28 = active_cohort(28).join(early, keys, "left").fillna({"clicks_28": 0})

cohort28.groupBy("at_risk").agg(
    F.count("*").alias("n"),
    F.avg("clicks_28").alias("avg_clicks"),
    F.expr("percentile_approx(clicks_28, 0.5)").alias("median_clicks")
).show()

10655280
+-------+-----+------------------+-------------+
|at_risk|    n|        avg_clicks|median_clicks|
+-------+-----+------------------+-------------+
|      1|12153|250.24010532378836|          152|
|      0|15385| 440.8748781280468|          298|
+-------+-----+------------------+-------------+

